# yt-clips Colab Worker
## Runtime -> T4 GPU
Run all cells. Worker listens for jobs from your Mac.

### Setup:
1. Create a Colab secret named `OPENROUTER_API_KEY` with your key
2. Or upload `.env` file to `yt-clips/` folder on Drive

In [ ]:
# CELL 1: Mount Drive + Load Env
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

import os, sys, time, subprocess
from pathlib import Path

for p in ["/content/drive/MyDrive/yt-clips", "/content/drive/My Drive/yt-clips"]:
    if Path(p).exists():
        os.chdir(p)
        print(f"OK: {p}")
        break
else:
    print("WARN: yt-clips not found on Drive, using /content")
    os.chdir("/content")

# Load .env from Drive if exists
if Path(".env").exists():
    for line in open(".env"):
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            k, v = line.split("=", 1)
            os.environ[k.strip()] = v.strip()
    print("Loaded .env from Drive")

# Load API key from Colab secrets (overrides .env)
try:
    from google.colab import userdata
    for key_name in ["OPENROUTER_API_KEY", "GOOGLE_API_KEY", "AI_API_KEY", "GROQ_API_KEY"]:
        val = userdata.get(key_name)
        if val:
            os.environ[key_name] = val
            print(f"Loaded {key_name} from secrets")
except:
    print("No Colab secrets found")

print("git pulling latest code...")
subprocess.run("git pull origin main 2>&1 | tail -3", shell=True)

In [ ]:
# CELL 2: Install Dependencies
def run(cmd, timeout=120):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
    if r.returncode != 0:
        err = r.stderr.strip()[-200:] if r.stderr else r.stdout.strip()[-200:]
        print(f"  WARN: {cmd[:60]}... ({err})")
    return r

print("System deps...")
run("apt-get install -y -qq aria2 ffmpeg openssh-client > /dev/null 2>&1")

print("Python deps...")
run("pip install -q yt-dlp faster-whisper rich PyYAML opencv-python-headless numpy "
    "filterpy scipy google-genai google-generativeai openai python-dotenv "
    "ultralytics torch --extra-index-url https://download.pytorch.org/whl/cu121")

gpu = subprocess.run("nvidia-smi --query-gpu=name --format=csv,noheader 2>/dev/null",
    shell=True, capture_output=True, text=True).stdout.strip()
print(f"GPU: {gpu or 'NONE! Use T4'}")

In [ ]:
# CELL 3: Start Worker + Tunnel
def kill_old():
    run("pkill -f 'python watcher.py' 2>/dev/null || true")
    run("pkill -f 'lt --port' 2>/dev/null || true")
    run("pkill -f serveo 2>/dev/null || true")
    run("fuser -k 5000/tcp 2>/dev/null || true")
    time.sleep(2)

for folder in ["input", "temp", "transcripts", "highlights", "shorts", "logs", "photos"]:
    Path(folder).mkdir(exist_ok=True)

kill_old()

# Start watcher
print("Starting watcher...")
subprocess.Popen([sys.executable, "watcher.py"], stdout=open("watcher.log","w"), stderr=subprocess.STDOUT)
time.sleep(3)

pid = subprocess.run("pgrep -f 'python watcher.py'", shell=True, capture_output=True, text=True).stdout.strip()
if pid:
    print(f"Watcher OK (PID: {pid.split()[0]})")
else:
    print("Watcher FAILED!")
    print(open("watcher.log").read().strip()[-300:])

# Start tunnel
print("Starting tunnel (serveo)...")
subprocess.Popen(
    ["ssh", "-o", "StrictHostKeyChecking=no", "-R", "80:localhost:5000", "serveo.net"],
    stdout=open("tunnel.log","w"), stderr=subprocess.STDOUT,
)
time.sleep(8)

# Extract URL
url = None
for i in range(20):
    time.sleep(2)
    for line in open("tunnel.log").read().splitlines():
        if "serveousercontent.com" in line or "https://" in line:
            for word in line.split():
                if "https://" in word:
                    url = word.strip().rstrip(",.")
                    break
    if url:
        break

if url:
    Path("colab_url.txt").write_text(url)
    print(f"Tunnel URL: {url}")
else:
    print("Tunnel URL not found. Log:")
    for l in open("tunnel.log").read().strip().splitlines()[-5:]:
        print(f"  {l}")

In [ ]:
# CELL 4: Monitor (keep running)
print("=" * 40)
print("  WORKER IS ONLINE!")
print("=" * 40)
print("On Mac:")
print('  python bridge.py "https://youtu.be/VIDEO_ID"')

try:
    last_pos = 0
    last_inode = None
    while True:
        time.sleep(10)
        try:
            st = Path("watcher.log").stat()
        except FileNotFoundError:
            continue
        if last_inode is not None and st.st_ino != last_inode:
            last_pos = 0
        last_inode = st.st_ino
        with open("watcher.log", "r") as f:
            f.seek(last_pos)
            for l in f.readlines():
                if l.strip():
                    print(l.strip())
            last_pos = f.tell()
except KeyboardInterrupt:
    print("Stopped.")